In [ ]:
!pip install torchmetrics

In [ ]:
import torch
import torchvision
from pathlib import Path
from torchmetrics.detection import MeanAveragePrecision
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

In [ ]:
DATA_ROOT = "/content/drive/MyDrive/datasets/player_football/set1"
NUM_CLASSES = 3
NUM_EPOCHS = 20
BATCH_SIZE = 4
LR = 0.005

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", DEVICE)

In [ ]:
class YOLOLabelDataset(Dataset):
    def __init__(self, root, split, transforms=None):
        self.img_dir = Path(root) / split / "images"
        self.lbl_dir = Path(root) / split / "labels"
        self.transforms = transforms
        self.images = sorted(self.img_dir.iterdir())

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_filepath = self.images[idx]
        base_filename = img_filepath.stem

        image = Image.open(img_filepath).convert("RGB")
        if self.transforms:
            img_tensor = self.transforms(image)

        bbox_coords = []
        labels = []

        img_w, img_h = image.size

        lbl_filepath = self.lbl_dir / (base_filename + ".txt")
        lines = Path(lbl_filepath).read_text().splitlines()
        
        for line in lines:
            parts = line.split()
            labels.append(int(parts[0]) + 1)
            bbox_xyxy = self.yolo_to_xyxy(parts[1:], img_w, img_h)
            bbox_coords.append(bbox_xyxy)
        target = {
            "boxes": torch.tensor(bbox_coords, dtype=torch.float32),
            "labels": torch.tensor(labels, dtype=torch.long)
        }
        return (img_tensor, target)

    def yolo_to_xyxy(self, bbox, img_w, img_h):
        cx, cy, bw, bh = [float(x) for x in bbox]
        x1 = (cx - bw/2) * img_w
        y1 = (cy - bh/2) * img_h
        x2 = (cx + bw/2) * img_w
        y2 = (cy + bh/2) * img_h

        return [x1, y1, x2, y2]

In [ ]:
train_dataset = YOLOLabelDataset(DATA_ROOT, "train", transforms=transforms.ToTensor())
val_dataset = YOLOLabelDataset(DATA_ROOT, "valid", transforms=transforms.ToTensor())

In [ ]:
def collate_fn(batch):
    return tuple(zip(*batch))

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

In [ ]:
model = fasterrcnn_resnet50_fpn(weights="DEFAULT")
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, NUM_CLASSES)
model.to(DEVICE)

In [ ]:
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=LR, momentum=0.9, weight_decay=0.005)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

In [ ]:
best_map = 0.0
for epoch in range(NUM_EPOCHS):
    model.train()
    total_loss = 0

    for images, targets in train_loader:
        images = [img.to(DEVICE) for img in images]
        targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]

        optimizer.zero_grad()
        loss_dict = model(images, targets)
        loss = sum(loss_dict.values())
        loss.backward()
        optimizer.step()
    
        total_loss += loss.item()

    avg_loss = total_loss/len(train_loader)
    scheduler.step()
    print(f"Epoch {epoch+1}/{NUM_EPOCHS}  loss: {avg_loss:.4f}  lr: {scheduler.get_last_lr()[0]:.5f}")

    model.eval()
    metric = MeanAveragePrecision()
    with torch.no_grad():
        for images, targets in val_loader:
            images = [img.to(DEVICE) for img in images]
            targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]

            predictions = model(images) 
            metric.update(predictions, targets)
    
    results = metric.compute()
    print(f"  val mAP@0.5:0.95: {results['map'].item():.4f}  mAP@0.5: {results['map_50'].item():.4f}")

    if(results['map_50'].item() > best_map):
        best_map = results['map_50'].item()
        torch.save(model.state_dict(), "best.pth")
        print(f"   checkpoint saved (mAP@0.5: {best_map:.4f})")